
# AI-Based Hospital Readmission Prediction System

# Feature Engineering


# Objective

The objective of this notebook is to prepare and transform the dataset
into a machine-learning-ready feature set.

# Feature Engineering Steps

- Load the cleaned raw dataset
- Create the binary readmission target
- Remove identifier columns
- Remove highly missing columns
- Create meaningful derived features
- Handle infinite values
- Review the final feature set
- Save the feature-engineered dataset

# Prediction Target

- 1 = Readmitted within 30 days
- 0 = Not readmitted within 30 days

# Important Note

Feature engineering is performed to create useful representations of the
available patient and healthcare information.

Features that are unavailable at the defined prediction point should not
be used in a real deployment system.

# Import Libraries

In [2]:
# Import pandas for data loading, manipulation, and analysis
import pandas as pd

# Import numpy for numerical operations and handling infinite values
import numpy as np

# Import os for creating folders and handling file paths
import os

# Confirm successful library import
print("All required libraries imported successfully.")

All required libraries imported successfully.


# Load Dataset

In [4]:
# Load the original UCI hospital diabetes dataset
df = pd.read_csv("diabetic_data.csv")

# Replace '?' with NaN so missing values can be handled correctly
df = df.replace("?", np.nan)

# Display the dataset dimensions
print("Dataset loaded successfully.")

# Display the first five rows to verify that the dataset loaded correctly
df.head()

Dataset loaded successfully.


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


# Check Dataset Shape

In [5]:
# Display the number of rows in the dataset
print("Number of Rows:", df.shape[0])

# Display the number of columns in the dataset
print("Number of Columns:", df.shape[1])

Number of Rows: 101766
Number of Columns: 50


# 1. Create the Binary Target

The original `readmitted` column contains three categories:

- `<30` = Readmitted within 30 days
- `>30` = Readmitted after 30 days
- `NO` = Not readmitted

For this project, the prediction task is binary classification.

Therefore:

- `<30` → 1
- `>30` → 0
- `NO` → 0

# Create Binary Target

In [6]:
# Create a binary target column for 30-day readmission
# '<30' is converted to 1 because it represents readmission within 30 days
# '>30' and 'NO' are converted to 0

df["readmitted_binary"] = (df["readmitted"] == "<30").astype(int)

# Display the original and binary target columns for verification
df[["readmitted", "readmitted_binary"]].head(10)

,readmitted,readmitted_binary
0,NO,0
1,>30,0
2,NO,0
3,NO,0
4,NO,0
5,>30,0
6,NO,0
7,>30,0
8,NO,0
9,NO,0


# Verify Target

In [7]:
# Count the number of records in each binary target class
target_counts = df["readmitted_binary"].value_counts().sort_index()

# Display the target counts
print("Binary Target Distribution:")
print(target_counts)

Binary Target Distribution:
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64


# Target Percentage

In [8]:
# Calculate the percentage of patients in each target class
target_percentage = (
    df["readmitted_binary"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

# Display the target percentages
print("Binary Target Percentage:")
print(target_percentage.round(2))

Binary Target Percentage:
readmitted_binary
0    88.84
1    11.16
Name: proportion, dtype: float64


# 2. Remove Identifier Columns

The following columns are identifiers rather than useful predictive features:

- `encounter_id`
- `patient_nbr`

These columns identify individual records or patients and should not be
used as predictive variables.

The original `readmitted` column is also removed because the new
`readmitted_binary` column is being used as the target.

# Remove IDs and Original Target

In [9]:
# Define identifier columns that should not be used for model training
id_columns = [
    "encounter_id",
    "patient_nbr"
]

# Remove identifier columns from the dataset
df = df.drop(columns=id_columns)

# Remove the original three-class target because the binary target is used
df = df.drop(columns=["readmitted"])

# Display the updated dataset shape
print("Dataset Shape After Removing IDs and Original Target:", df.shape)

Dataset Shape After Removing IDs and Original Target: (101766, 48)


# Check Remaining Columns

In [10]:
# Display all columns remaining after removing identifiers and original target
print("Remaining Columns:")

# Print the column names one by one
for column in df.columns:
    print(column)

Remaining Columns:
race
gender
age
weight
admission_type_id
discharge_disposition_id
admission_source_id
time_in_hospital
payer_code
medical_specialty
num_lab_procedures
num_procedures
num_medications
number_outpatient
number_emergency
number_inpatient
diag_1
diag_2
diag_3
number_diagnoses
max_glu_serum
A1Cresult
metformin
repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
glipizide
glyburide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
examide
citoglipton
insulin
glyburide-metformin
glipizide-metformin
glimepiride-pioglitazone
metformin-rosiglitazone
metformin-pioglitazone
change
diabetesMed
readmitted_binary


# 3. Missing Value Analysis

Some variables contain a large percentage of missing values.

Columns with extremely high missingness provide limited information
and can make the model unnecessarily complex.

In this project, columns with more than 90% missing values will be
identified and removed.

# Calculate Missing Value Percentage

In [11]:
# Calculate the percentage of missing values in every column
missing_percentage = df.isnull().mean() * 100

# Sort the columns from highest to lowest missing percentage
missing_percentage = missing_percentage.sort_values(ascending=False)

# Display the missing-value percentages
missing_percentage.head(20).round(2)

weight               96.86
max_glu_serum        94.75
A1Cresult            83.28
medical_specialty    49.08
payer_code           39.56
race                  2.23
diag_3                1.40
diag_2                0.35
diag_1                0.02
tolbutamide           0.00
pioglitazone          0.00
rosiglitazone         0.00
acarbose              0.00
miglitol              0.00
troglitazone          0.00
examide               0.00
tolazamide            0.00
glipizide             0.00
citoglipton           0.00
insulin               0.00
dtype: float64

# Identify Highly Missing Columns

In [12]:
# Set the missing-value threshold to 90 percent
missing_threshold = 90

# Identify columns where more than 90 percent of values are missing
high_missing_columns = missing_percentage[
    missing_percentage > missing_threshold
].index.tolist()

# Display the identified columns
print("Columns with More Than 90% Missing Values:")

# Print the columns
print(high_missing_columns)

Columns with More Than 90% Missing Values:
['weight', 'max_glu_serum']


# Count Highly Missing Columns

In [14]:
# Count the number of columns having more than 90% missing values
high_missing_count = len(high_missing_columns)

# Display the number of highly missing columns
print(
    "Number of Columns with More Than 90% Missing Values:",
    high_missing_count
)

Number of Columns with More Than 90% Missing Values: 2


# Code: Remove Highly Missing Columns

In [15]:
# Remove columns with more than 90% missing values
df = df.drop(columns=high_missing_columns)

# Display the new dataset shape after removing sparse columns
print("Dataset Shape After Removing Highly Missing Columns:", df.shape)

Dataset Shape After Removing Highly Missing Columns: (101766, 46)


# Verify Missing Values

In [16]:
# Calculate the remaining missing values in every column
remaining_missing = df.isnull().sum()

# Keep only columns that still contain missing values
remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)

# Display the remaining missing-value counts
remaining_missing.head(20)

A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

# 4. Create Derived Features

New features can be created by combining existing information.

The following features will be created:

### 1. total_prior_visits

Combines:
- Previous outpatient visits
- Previous emergency visits
- Previous inpatient visits

### 2. total_procedures

Combines:
- Laboratory procedures
- Other procedures

### 3. medications_per_diagnosis

Represents the number of medications relative to the number of diagnoses.

These features may provide a more compact representation of patient
healthcare utilization and treatment complexity.

# Create Total Prior Visits

In [17]:
# Create a new feature representing total previous healthcare visits
# Add outpatient, emergency, and inpatient visits together
df["total_prior_visits"] = (
    df["number_outpatient"].fillna(0)
    + df["number_emergency"].fillna(0)
    + df["number_inpatient"].fillna(0)
)

# Display the new feature
df[[
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "total_prior_visits"
]].head()

,number_outpatient,number_emergency,number_inpatient,total_prior_visits
0,0,0,0,0
1,0,0,0,0
2,2,0,1,3
3,0,0,0,0
4,0,0,0,0


# Create Total Procedures

In [18]:
# Create a new feature representing the total number of procedures
# Add laboratory procedures and other procedures together
df["total_procedures"] = (
    df["num_lab_procedures"].fillna(0)
    + df["num_procedures"].fillna(0)
)

# Display the new feature
df[[
    "num_lab_procedures",
    "num_procedures",
    "total_procedures"
]].head()

,num_lab_procedures,num_procedures,total_procedures
0,41,0,41
1,59,0,59
2,11,5,16
3,44,1,45
4,51,0,51


# Create Medications per Diagnosis

In [19]:
# Calculate the number of medications relative to the number of diagnoses
# Replace zero diagnoses with NaN temporarily to avoid division by zero
diagnosis_denominator = df["number_diagnoses"].replace(0, np.nan)

# Create the new feature
df["medications_per_diagnosis"] = (
    df["num_medications"] / diagnosis_denominator
)

# Display the new feature
df[[
    "num_medications",
    "number_diagnoses",
    "medications_per_diagnosis"
]].head()

,num_medications,number_diagnoses,medications_per_diagnosis
0,1,1,1.000000
1,18,9,2.000000
2,13,6,2.166667
3,16,7,2.285714
4,8,5,1.600000


# Check New Features

In [20]:
# Display descriptive statistics for the newly created features
new_features = [
    "total_prior_visits",
    "total_procedures",
    "medications_per_diagnosis"
]

# Generate summary statistics for the new features
df[new_features].describe()

,total_prior_visits,total_procedures,medications_per_diagnosis
count,101766.000000,101766.000000,101766.000000
mean,1.202759,44.435371,2.282074
std,2.291781,19.846605,1.320756
min,0.000000,1.000000,0.111111
25%,0.000000,33.000000,1.444444
50%,0.000000,45.000000,2.000000
75%,2.000000,58.000000,2.777778
max,80.000000,135.000000,24.000000


# 5. Handle Infinite Values

The `medications_per_diagnosis` feature involves division.

If the denominator is zero, the calculation can produce infinite values.

Infinite values are not suitable for machine-learning algorithms.

Therefore, all positive and negative infinite values will be converted
to missing values (NaN).

# Check Infinite Values

In [21]:
# Identify infinite values in numerical columns
infinite_values = np.isinf(
    df.select_dtypes(include=np.number)
).sum()

# Display only columns containing infinite values
infinite_values = infinite_values[
    infinite_values > 0
]

# Display the result
print("Infinite Values Before Handling:")
print(infinite_values)

Infinite Values Before Handling:
Series([], dtype: int64)


# Replace Infinite Values

In [22]:
# Replace positive and negative infinite values with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Verify that infinite values have been removed
infinite_values_after = np.isinf(
    df.select_dtypes(include=np.number)
).sum().sum()

# Display the total number of remaining infinite values
print(
    "Total Infinite Values After Handling:",
    infinite_values_after
)

Total Infinite Values After Handling: 0


# Check New Feature Missing Values

In [23]:
# Check missing values in the newly created features
new_feature_missing = df[new_features].isnull().sum()

# Display missing-value counts
print("Missing Values in New Features:")
print(new_feature_missing)

Missing Values in New Features:
total_prior_visits           0
total_procedures             0
medications_per_diagnosis    0
dtype: int64


# 6. Review Numerical Features

Before saving the feature-engineered dataset, we will review the
available numerical features.

This helps confirm that the newly created variables have been added
correctly and that the dataset is ready for the next stage.

# Identify Numerical Features

In [24]:
# Identify all numerical columns in the feature-engineered dataset
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

# Display the numerical features
print("Numerical Features:")

# Print each numerical feature
for column in numerical_columns:
    print(column)

Numerical Features:
admission_type_id
discharge_disposition_id
admission_source_id
time_in_hospital
num_lab_procedures
num_procedures
num_medications
number_outpatient
number_emergency
number_inpatient
number_diagnoses
readmitted_binary
total_prior_visits
total_procedures
medications_per_diagnosis


# Identify Categorical Features

In [25]:
# Identify all categorical columns in the feature-engineered dataset
categorical_columns = df.select_dtypes(
    exclude=np.number
).columns.tolist()

# Display the categorical features
print("Categorical Features:")

# Print each categorical feature
for column in categorical_columns:
    print(column)

Categorical Features:
race
gender
age
payer_code
medical_specialty
diag_1
diag_2
diag_3
A1Cresult
metformin
repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
glipizide
glyburide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
examide
citoglipton
insulin
glyburide-metformin
glipizide-metformin
glimepiride-pioglitazone
metformin-rosiglitazone
metformin-pioglitazone
change
diabetesMed


# Check Feature Count

In [26]:
# Count the total number of features including the target column
total_columns = len(df.columns)

# Count the numerical features
total_numerical = len(numerical_columns)

# Count the categorical features
total_categorical = len(categorical_columns)

# Display the feature counts
print("Total Columns:", total_columns)
print("Numerical Columns:", total_numerical)
print("Categorical Columns:", total_categorical)

Total Columns: 49
Numerical Columns: 15
Categorical Columns: 34


# 7. Final Missing Value Summary

Missing values will not be manually filled in this notebook.

The actual imputation will be performed inside the machine-learning
preprocessing pipeline in Notebook 05.

This prevents information leakage because the imputation parameters
will be learned from the training data only.

# Final Missing Value Check

In [27]:
# Calculate the number of missing values in every column
final_missing = df.isnull().sum()

# Keep only columns that contain missing values
final_missing = final_missing[
    final_missing > 0
].sort_values(ascending=False)

# Display the remaining missing values
print("Remaining Missing Values:")
print(final_missing.head(20))

Remaining Missing Values:
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


# 8. Final Feature Dataset

The feature-engineered dataset now contains:

- Original useful patient features
- Healthcare utilization features
- Treatment-related features
- Newly created derived features
- Binary readmission target

The dataset will now be saved for use in the model-training notebook.

# Display Final Dataset

In [28]:
# Display the first five rows of the final feature-engineered dataset
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,total_prior_visits,total_procedures,medications_per_diagnosis
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,0,0,41,1.000000
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,No,No,No,Ch,Yes,0,0,59,2.000000
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,Yes,0,3,16,2.166667
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,No,No,No,No,Ch,Yes,0,0,45,2.285714
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,No,No,No,No,Ch,Yes,0,0,51,1.600000


# Final Dataset Shape

In [29]:
# Display the final number of rows
print("Final Number of Rows:", df.shape[0])

# Display the final number of columns
print("Final Number of Columns:", df.shape[1])

Final Number of Rows: 101766
Final Number of Columns: 49


# Verify Target Exists

In [30]:
# Check whether the binary target column exists in the final dataset
if "readmitted_binary" in df.columns:
    print("readmitted_binary target column is present.")
else:
    print("ERROR: Target column is missing.")

readmitted_binary target column is present.


# Verify ID Columns Removed

In [31]:
# Check whether identifier columns are still present
remaining_id_columns = [
    column
    for column in ["encounter_id", "patient_nbr"]
    if column in df.columns
]

# Display the result
if len(remaining_id_columns) == 0:
    print("Identifier columns were successfully removed.")
else:
    print("Remaining identifier columns:", remaining_id_columns)

Identifier columns were successfully removed.


# Create Artifacts Folder

In [32]:
# Define the folder where project artifacts and processed data will be stored
artifacts_folder = "artifacts"

# Create the artifacts folder if it does not already exist
os.makedirs(artifacts_folder, exist_ok=True)

# Confirm that the folder exists
print("Artifacts folder is ready.")

Artifacts folder is ready.


# Save Feature-Engineered Dataset

In [33]:
# Define the output path for the feature-engineered dataset
output_path = os.path.join(
    artifacts_folder,
    "feature_engineered_data.csv"
)

# Save the feature-engineered dataset as a CSV file
# index=False prevents pandas from creating an extra index column
df.to_csv(
    output_path,
    index=False
)

# Display the saved file location
print("Feature-engineered dataset saved successfully:")
print(output_path)

Feature-engineered dataset saved successfully:
artifacts\feature_engineered_data.csv


# Verify Saved File

In [34]:
# Check whether the feature-engineered dataset file exists
file_exists = os.path.exists(output_path)

# Display the verification result
print("File Exists:", file_exists)

File Exists: True


# Reload Saved Dataset

In [35]:
# Reload the saved feature-engineered dataset
# This verifies that the saved file can be read correctly
df_check = pd.read_csv(output_path)

# Display the shape of the reloaded dataset
print("Reloaded Dataset Shape:", df_check.shape)

Reloaded Dataset Shape: (101766, 49)


C:\Users\Asus\AppData\Local\Temp\ipykernel_12216\3041708043.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(output_path)


# Verify Reloaded Data

In [36]:
# Display the first five rows of the reloaded dataset
df_check.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,total_prior_visits,total_procedures,medications_per_diagnosis
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,0,0,41,1.000000
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,No,No,No,Ch,Yes,0,0,59,2.000000
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,Yes,0,3,16,2.166667
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,No,No,No,No,Ch,Yes,0,0,45,2.285714
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,No,No,No,No,Ch,Yes,0,0,51,1.600000


# Verify Target Distribution Again

In [37]:
# Check the target distribution after saving and reloading the dataset
final_target_distribution = (
    df_check["readmitted_binary"]
    .value_counts()
    .sort_index()
)

# Display the final target distribution
print("Final Target Distribution:")
print(final_target_distribution)

Final Target Distribution:
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64


# Final Feature List

In [38]:
# Create a list containing all columns in the final dataset
final_features = df_check.columns.tolist()

# Display the total number of final columns
print("Total Final Columns:", len(final_features))

# Display all final columns
print("\nFinal Feature List:")

# Print each feature name
for feature in final_features:
    print(feature)

Total Final Columns: 49

Final Feature List:
race
gender
age
admission_type_id
discharge_disposition_id
admission_source_id
time_in_hospital
payer_code
medical_specialty
num_lab_procedures
num_procedures
num_medications
number_outpatient
number_emergency
number_inpatient
diag_1
diag_2
diag_3
number_diagnoses
A1Cresult
metformin
repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
glipizide
glyburide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
examide
citoglipton
insulin
glyburide-metformin
glipizide-metformin
glimepiride-pioglitazone
metformin-rosiglitazone
metformin-pioglitazone
change
diabetesMed
readmitted_binary
total_prior_visits
total_procedures
medications_per_diagnosis


# 9. Prediction Point and Leakage Consideration

The dataset contains variables that describe different stages of the
hospital encounter.

For this project, the feature set should be interpreted according to
the defined prediction point.

Features that are not available at the prediction point should be
excluded in a real-time implementation.

The current project uses retrospective hospital data for model
development. Therefore, the availability of each feature must be
validated before using the model in a live clinical workflow.

# Feature Engineering Summary

The following derived features were created:

# total_prior_visits
Sum of previous outpatient, emergency, and inpatient visits.

# total_procedures
Sum of laboratory procedures and other procedures.

# medications_per_diagnosis
Number of medications divided by the number of diagnoses.

Highly sparse columns were removed based on a 90% missing-value
threshold.

Missing-value imputation is intentionally deferred to the machine-learning
pipeline so that imputation parameters are learned only from training data.

# Final Summary

In [39]:
# Calculate the final number of patients
final_rows = df_check.shape[0]

# Calculate the final number of columns
final_columns = df_check.shape[1]

# Calculate the number of 30-day readmission cases
readmission_cases = int(
    df_check["readmitted_binary"].sum()
)

# Calculate the 30-day readmission percentage
readmission_rate = (
    df_check["readmitted_binary"].mean() * 100
)

# Display the final feature engineering summary
print("==========================================")
print("FEATURE ENGINEERING SUMMARY")
print("==========================================")
print("Total Patients:", final_rows)
print("Total Columns:", final_columns)
print("30-Day Readmission Cases:", readmission_cases)
print(
    "30-Day Readmission Rate:",
    round(readmission_rate, 2),
    "%"
)
print("==========================================")

FEATURE ENGINEERING SUMMARY
Total Patients: 101766
Total Columns: 49
30-Day Readmission Cases: 11357
30-Day Readmission Rate: 11.16 %


# Conclusion

Feature Engineering has been completed successfully.

The dataset has been transformed by:

- Creating a binary 30-day readmission target
- Removing identifier columns
- Removing highly sparse columns
- Creating total prior healthcare visits
- Creating total procedures
- Creating medications per diagnosis
- Handling infinite values
- Reviewing numerical and categorical features
- Saving the final feature-engineered dataset

The resulting dataset will be used in the next stage:

**Model Training and Comparison**

# Completion Check

In [40]:
# Verify that the saved feature-engineered dataset exists
if os.path.exists(output_path):

    # Print successful completion message
    print("Notebook 04 completed successfully.")

    # Display the location of the saved dataset
    print("Saved File:", output_path)

else:

    # Print an error message if the file was not saved
    print("ERROR: Feature-engineered dataset was not saved.")

Notebook 04 completed successfully.
Saved File: artifacts\feature_engineered_data.csv
